# BERT Deep Analysis: Classification + Attention + Positional Encoding
## Extracting and visualizing internal mechanisms from a pretrained BERT model

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

import matplotlib.pyplot as plt
%matplotlib inline

import seaborn as sns
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.manifold import TSNE

## STEP 1: Load Data and BERT Model

In [ ]:
# Load fake and real news datasets
df_fake = pd.read_csv("C:/Users/HJ25ZC/OneDrive - Aalborg Universitet/Documents/Kurser/AI og data/Fake_True/Fake.csv")
df_real = pd.read_csv("C:/Users/HJ25ZC/OneDrive - Aalborg Universitet/Documents/Kurser/AI og data/Fake_True/True.csv")

df_fake['label'] = 1  # Fake
df_real['label'] = 0  # Real

df = pd.concat([df_fake[:1000], df_real[:1000]], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Dataset shape: {df.shape}")
print(f"Real news: {(df['label']==0).sum()}, Fake news: {(df['label']==1).sum()}")

In [ ]:
# Setup device and load BERT model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

# Load pretrained BERT with eager attention
checkpoint_path = 'Bert_checkpoints/checkpoint_epoch_0_batch_499.pt'
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2,
    attn_implementation='eager',
    output_attentions=True,
    output_hidden_states=True
)

checkpoint = torch.load(checkpoint_path, map_location=device)
if 'model_state_dict' in checkpoint:
    state_dict = checkpoint['model_state_dict']
else:
    state_dict = checkpoint

model.load_state_dict(state_dict, strict=False)
model.to(device)
model.eval()

print("✓ Model loaded successfully")
print(f"Model config: {model.config}")

In [ ]:
# Prepare data
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

texts = df['title'].astype(str).tolist()
labels = df['label'].values

# Split into train/validation
text_train, text_val, label_train, label_val = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

# Tokenize
val_encodings = tokenizer(
    text_val,
    truncation=True,
    max_length=128,
    padding=True,
    return_tensors='pt'
)

val_input_ids = val_encodings['input_ids'].to(device)
val_attention_mask = val_encodings['attention_mask'].to(device)
val_labels = torch.tensor(label_val, dtype=torch.long).to(device)

print(f"Validation set: {len(text_val)} samples")
print(f"Input shape: {val_input_ids.shape}")
print(f"Max sequence length: {val_input_ids.shape[1]}")

## STEP 2: Classification Results

In [ ]:
# Run inference and collect predictions + model outputs
all_predictions = []
all_logits = []
all_attentions = []  # List of tuples (one per layer)
all_hidden_states = []  # Hidden states from all layers

with torch.no_grad():
    outputs = model(
        input_ids=val_input_ids,
        attention_mask=val_attention_mask,
        output_attentions=True,
        output_hidden_states=True
    )
    
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1).cpu().numpy()
    attentions = outputs.attentions  # Tuple of length num_layers, each (batch, heads, seq, seq)
    hidden_states = outputs.hidden_states  # Tuple of length num_layers+1
    
    all_predictions = predictions
    all_logits = logits.cpu().numpy()
    all_attentions = [att.cpu().numpy() for att in attentions]
    all_hidden_states = [hs.cpu().numpy() for hs in hidden_states]

print(f"Predictions shape: {all_predictions.shape}")
print(f"Number of BERT layers: {len(all_attentions)}")
print(f"Attention shape per layer: {all_attentions[0].shape} (batch, heads, seq_len, seq_len)")
print(f"Hidden states per layer: {all_hidden_states[0].shape}")

In [ ]:
# Calculate metrics
accuracy = accuracy_score(label_val, all_predictions)
cm = confusion_matrix(label_val, all_predictions)

print("="*60)
print("CLASSIFICATION RESULTS")
print("="*60)
print(f"\nAccuracy: {accuracy:.4f}\n")
print(f"Confusion Matrix:\n{cm}\n")
print("Classification Report:")
print(classification_report(label_val, all_predictions, target_names=['Real', 'Fake']))

In [ ]:
# Visualize classification results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=True)
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')
axes[0].set_xticklabels(['Real', 'Fake'])
axes[0].set_yticklabels(['Real', 'Fake'])

# Normalized
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Greens', ax=axes[1], cbar=True)
axes[1].set_title('Confusion Matrix (Normalized)', fontweight='bold')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')
axes[1].set_xticklabels(['Real', 'Fake'])
axes[1].set_yticklabels(['Real', 'Fake'])

plt.tight_layout()
plt.show()

## STEP 3: Extract and Analyze BERT's Internal Mechanisms

### 3.1 Attention Allocation Across Layers

In [ ]:
# Analyze attention patterns across all 12 BERT layers
num_layers = len(all_attentions)
num_heads = all_attentions[0].shape[1]
seq_len = all_attentions[0].shape[2]

print(f"BERT Model Details:")
print(f"  Layers: {num_layers}")
print(f"  Heads per layer: {num_heads}")
print(f"  Sequence length: {seq_len}")
print(f"  Samples: {all_attentions[0].shape[0]}")

# Average attention across samples and heads for each layer
layer_avg_attentions = []
for layer_idx in range(num_layers):
    layer_attn = all_attentions[layer_idx]  # (batch, heads, seq, seq)
    avg_attn = layer_attn.mean(axis=(0, 1))  # Average over batch and heads
    layer_avg_attentions.append(avg_attn)

# Visualize attention across layers
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for layer_idx in range(num_layers):
    im = axes[layer_idx].imshow(layer_avg_attentions[layer_idx], cmap='viridis', aspect='auto')
    axes[layer_idx].set_title(f'Layer {layer_idx}', fontweight='bold')
    axes[layer_idx].set_xlabel('Attended Token')
    axes[layer_idx].set_ylabel('Query Token')
    plt.colorbar(im, ax=axes[layer_idx])

plt.suptitle('Average Attention Across All 12 BERT Layers', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

### 3.2 Multi-Head Attention Patterns in Final Layer

In [ ]:
# Visualize all 12 attention heads from the last layer
last_layer_attn = all_attentions[-1]  # (batch, 12, seq, seq)
avg_per_head = last_layer_attn.mean(axis=0)  # Average over batch: (12, seq, seq)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for head_idx in range(num_heads):
    im = axes[head_idx].imshow(avg_per_head[head_idx], cmap='RdYlBu_r', aspect='auto', vmin=0, vmax=0.3)
    axes[head_idx].set_title(f'Head {head_idx}', fontweight='bold', fontsize=10)
    axes[head_idx].set_xlabel('Token (Attended)', fontsize=8)
    axes[head_idx].set_ylabel('Token (Query)', fontsize=8)
    plt.colorbar(im, ax=axes[head_idx])

plt.suptitle('Multi-Head Attention in BERT Final Layer\n(Each Head Focuses on Different Patterns)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.3 Attention Masking Impact

In [ ]:
# Show how attention masking affects the [CLS] token (classification token)
# [CLS] token is at position 0 and is used for classification

cls_attentions_per_layer = []
for layer_idx in range(num_layers):
    layer_attn = all_attentions[layer_idx]  # (batch, heads, seq, seq)
    # Get attention from [CLS] token to all other tokens
    cls_attn = layer_attn[:, :, 0, :]  # (batch, heads, seq) - what CLS attends to
    avg_cls_attn = cls_attn.mean(axis=(0, 1))  # (seq,) - average across batch and heads
    cls_attentions_per_layer.append(avg_cls_attn)

# Visualize [CLS] attention across layers
fig, axes = plt.subplots(3, 4, figsize=(18, 10))
axes = axes.flatten()

for layer_idx in range(num_layers):
    cls_logits = cls_attentions_per_layer[layer_idx]
    actual_seq_len = val_attention_mask[0].cpu().numpy().sum()  # Actual sequence length (excluding padding)
    
    # Mark padding positions
    colors = ['red' if i >= actual_seq_len else 'blue' for i in range(seq_len)]
    axes[layer_idx].bar(range(seq_len), cls_logits, color=colors, alpha=0.7)
    axes[layer_idx].axvline(actual_seq_len-1, color='orange', linestyle='--', linewidth=2, label='Padding starts')
    axes[layer_idx].set_title(f'Layer {layer_idx} - [CLS] Token Attention', fontweight='bold')
    axes[layer_idx].set_xlabel('Token Position')
    axes[layer_idx].set_ylabel('Attention Weight')
    axes[layer_idx].set_ylim([0, cls_attentions_per_layer[layer_idx].max() * 1.2])

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='blue', alpha=0.7, label='Valid Tokens'),
                    Patch(facecolor='red', alpha=0.7, label='Padding (Masked)')]
fig.legend(handles=legend_elements, loc='upper center', ncol=2, fontsize=11)

plt.suptitle('[CLS] Token Attention Across BERT Layers\n(Red bars = tokens masked by attention mask)', 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

print(f"\nAttention Masking Analysis:")
print(f"Actual text tokens: {int(actual_seq_len)} (including [CLS] and [SEP])")
print(f"Padding tokens: {seq_len - int(actual_seq_len)}")
print(f"Total sequence length: {seq_len}")

### 3.4 Positional Encoding Analysis

In [ ]:
# Extract positional encodings from BERT's embedding layer
pos_embedding = model.bert.embeddings.position_embeddings.weight.cpu().detach().numpy()
token_embedding_dim = pos_embedding.shape[1]
max_pos = pos_embedding.shape[0]

print(f"Positional Encoding Details:")
print(f"  Dimension: {token_embedding_dim}")
print(f"  Max positions: {max_pos}")
print(f"  Shape: {pos_embedding.shape}")

# Visualize positional encoding pattern
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Heatmap of positional encodings
im = axes[0, 0].imshow(pos_embedding[:128, :], cmap='RdBu', aspect='auto')  # First 128 positions
axes[0, 0].set_title('BERT Positional Encoding Heatmap', fontweight='bold')
axes[0, 0].set_xlabel('Encoding Dimension')
axes[0, 0].set_ylabel('Position')
plt.colorbar(im, ax=axes[0, 0])

# 2. L2 norm of positional encodings across positions
pos_norms = np.linalg.norm(pos_embedding, axis=1)
axes[0, 1].plot(pos_norms, linewidth=2, color='darkblue')
axes[0, 1].fill_between(range(len(pos_norms)), pos_norms, alpha=0.3)
axes[0, 1].set_title('L2 Norm of Positional Encodings', fontweight='bold')
axes[0, 1].set_xlabel('Position in Sequence')
axes[0, 1].set_ylabel('L2 Norm')
axes[0, 1].grid(alpha=0.3)

# 3. Pairwise similarity between positional encodings
pos_sim = np.corrcoef(pos_embedding[:128])  # Correlation between position embeddings
im = axes[1, 0].imshow(pos_sim, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)
axes[1, 0].set_title('Positional Encoding Similarity (Correlation)', fontweight='bold')
axes[1, 0].set_xlabel('Position')
axes[1, 0].set_ylabel('Position')
plt.colorbar(im, ax=axes[1, 0])

# 4. First few dimensions of positional encoding
for dim_idx in range(min(6, token_embedding_dim)):
    axes[1, 1].plot(pos_embedding[:seq_len, dim_idx], label=f'Dim {dim_idx}', alpha=0.7)
axes[1, 1].set_title('First 6 Dimensions of Positional Encoding', fontweight='bold')
axes[1, 1].set_xlabel('Position in Sequence')
axes[1, 1].set_ylabel('Encoding Value')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 3.5 Similarity Matrices from Hidden States

In [ ]:
# Extract embeddings: input embeddings, middle layers, and final layer
# hidden_states indices: 0=input, 1-12=after each of 12 layers

# Take first sample for analysis
sample_idx = 0
actual_seq_len = int(val_attention_mask[sample_idx].sum().item())

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

layer_indices = [0, 3, 6, 9, 11, 12]  # Input, early, middle, late, final
layer_names = ['Input\n(Embedding)', 'Layer 3', 'Layer 6', 'Layer 9', 'Layer 11', 'Output\n(Final Layer)']

for idx, (layer_idx, layer_name) in enumerate(zip(layer_indices, layer_names)):
    # Get hidden state for this sample at this layer
    hidden = all_hidden_states[layer_idx][sample_idx]  # (seq_len, hidden_dim)
    
    # Compute similarity matrix (dot product of normalized vectors)
    hidden_norm = hidden / (np.linalg.norm(hidden, axis=1, keepdims=True) + 1e-8)
    similarity = hidden_norm @ hidden_norm.T  # (seq, seq)
    
    # Crop to actual sequence length for clarity
    similarity_cropped = similarity[:actual_seq_len, :actual_seq_len]
    
    im = axes[idx].imshow(similarity_cropped, cmap='viridis', aspect='auto', vmin=-1, vmax=1)
    axes[idx].set_title(f'{layer_name}\nSimilarity Matrix', fontweight='bold')
    axes[idx].set_xlabel('Token (Attended)')
    axes[idx].set_ylabel('Token (Query)')
    plt.colorbar(im, ax=axes[idx])

plt.suptitle('Embedding Evolution Through BERT Layers\n(Token-to-Token Similarity)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Sample analysis details:")
print(f"  Sample index: {sample_idx}")
print(f"  Actual tokens: {actual_seq_len}")
print(f"  Hidden dimension: {all_hidden_states[-1].shape[-1]}")

### 3.6 [CLS] Token Representation Across Layers

In [ ]:
# Analyze how [CLS] token representation evolves through layers
cls_representations = []
for layer_idx in range(len(all_hidden_states)):
    cls_rep = all_hidden_states[layer_idx][:, 0, :]  # (batch, hidden_dim) - [CLS] token
    cls_representations.append(cls_rep)

# Compute similarity of [CLS] representation between consecutive layers
cls_similarities = []
for i in range(len(cls_representations)-1):
    cls_prev = cls_representations[i]  # (batch, hidden_dim)
    cls_curr = cls_representations[i+1]  # (batch, hidden_dim)
    
    # Normalize
    cls_prev_norm = cls_prev / (np.linalg.norm(cls_prev, axis=1, keepdims=True) + 1e-8)
    cls_curr_norm = cls_curr / (np.linalg.norm(cls_curr, axis=1, keepdims=True) + 1e-8)
    
    # Cosine similarity
    sim = (cls_prev_norm * cls_curr_norm).sum(axis=1)
    cls_similarities.append(sim)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: [CLS] representation change across layers
average_sims = [sim.mean() for sim in cls_similarities]
axes[0].plot(range(1, len(average_sims)+1), average_sims, marker='o', linewidth=2, markersize=8, color='darkblue')
axes[0].fill_between(range(1, len(average_sims)+1), average_sims, alpha=0.3)
axes[0].set_title('[CLS] Token Representation Similarity Across Layers', fontweight='bold')
axes[0].set_xlabel('Layer Transition')
axes[0].set_ylabel('Average Cosine Similarity')
axes[0].set_xticks(range(1, len(average_sims)+1))
axes[0].grid(alpha=0.3)
axes[0].set_ylim([0, 1.05])

# Plot 2: Distribution of similarities across samples
box_data = [sim for sim in cls_similarities]
bp = axes[1].boxplot(box_data, labels=[f'L{i}→{i+1}' for i in range(len(box_data))], patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')
axes[1].set_title('[CLS] Representation Change Distribution', fontweight='bold')
axes[1].set_ylabel('Cosine Similarity')
axes[1].set_xlabel('Layer Transition')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"[CLS] Token Representation Analysis:")
print(f"Average similarity between consecutive layers:")
for i, sim in enumerate(average_sims):
    print(f"  Layer {i} → {i+1}: {sim:.4f}")

### 3.7 Combined Analysis: Attention + Masking + Positional Encoding

In [ ]:
# Show how attention, masking, and positional encoding work together
# Analyze one sample in detail

sample_idx = 0
actual_seq_len = int(val_attention_mask[sample_idx].sum().item())

fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(3, 4, hspace=0.3, wspace=0.3)

# Row 1: Attention masks across layers
for layer_idx in range(4):
    ax = fig.add_subplot(gs[0, layer_idx])
    # Visualize how attention is distributed - attending to real vs padding tokens
    layer_attn = all_attentions[layer_idx][sample_idx]  # (12 heads, seq, seq)
    avg_attn = layer_attn.mean(axis=0)  # (seq, seq)
    
    im = ax.imshow(avg_attn, cmap='viridis', aspect='auto')
    ax.axhline(actual_seq_len-0.5, color='red', linestyle='--', linewidth=1.5, label='Padding start')
    ax.axvline(actual_seq_len-0.5, color='red', linestyle='--', linewidth=1.5)
    ax.set_title(f'Layer {layer_idx}\nAttention Pattern', fontweight='bold', fontsize=9)
    ax.set_xlabel('Attended', fontsize=8)
    ax.set_ylabel('Query', fontsize=8)

# Row 2: Positional encoding contribution
pos_enc = pos_embedding[:seq_len]
token_enc = model.bert.embeddings.word_embeddings.weight.cpu().detach().numpy()[val_input_ids[sample_idx]]

for pos_idx, target_pos in enumerate([0, 5, 10, 20]):  # Different positions
    ax = fig.add_subplot(gs[1, pos_idx])
    if target_pos < actual_seq_len:
        pos_contribution = pos_enc[target_pos]
        ax.bar(range(min(50, len(pos_contribution))), pos_contribution[:50], color='purple', alpha=0.7)
        ax.set_title(f'Pos {target_pos}\nEncoding (First 50 dims)', fontweight='bold', fontsize=9)
        ax.set_xlabel('Dimension', fontsize=8)
        ax.set_ylabel('Value', fontsize=8)
    else:
        ax.text(0.5, 0.5, '[PADDING]', ha='center', va='center', fontsize=12, 
                bbox=dict(boxstyle='round', facecolor='red', alpha=0.3))
        ax.axis('off')

# Row 3: Summary statistics
ax_summary = fig.add_subplot(gs[2, :2])
ax_summary.text(0.05, 0.95, f'Analysis Summary', fontsize=12, fontweight='bold', 
                transform=ax_summary.transAxes, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
summary_text = f"""
Attention Masking: ✓ Applied
  • Valid tokens (0-{actual_seq_len-1}): Attend to each other
  • Padding tokens ({actual_seq_len}-{seq_len-1}): Ignored by attention
  • Attention mask: {val_attention_mask[sample_idx].cpu().numpy()[:actual_seq_len+2]}

Positional Encoding: ✓ Added
  • Method: Sinusoidal (sin/cos patterns)
  • Dimension: {token_embedding_dim}
  • Applied to all positions (0-{seq_len-1})

Layers Analyzed: 12 Transformer Layers
  • Multi-head attention (12 heads each)
  • Positional encoding + token embedding → input
  • Each layer refines token representations
"""
ax_summary.text(0.05, 0.80, summary_text, fontsize=9, transform=ax_summary.transAxes, 
                verticalalignment='top', family='monospace')
ax_summary.axis('off')

# Key insight
ax_insight = fig.add_subplot(gs[2, 2:])
insight_text = f"""
Key Insights from BERT Analysis:

1. Attention Masking:
   Padding tokens have near-zero attention weight
   Model focuses on actual content tokens

2. Positional Encoding:
   Sinusoidal patterns allow relative position learning
   Different dimensions capture different frequencies

3. Layer Evolution:
   Early layers: Capture local patterns
   Middle layers: Build semantic relationships
   Final layers: Aggregate for classification

4. [CLS] Token:
   Starts as random initialization
   Becomes document representation through layers
   Used for classification decision
"""
ax_insight.text(0.05, 0.95, insight_text, fontsize=10, transform=ax_insight.transAxes, 
                verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
ax_insight.axis('off')

plt.suptitle('Complete BERT Analysis: Attention + Masking + Positional Encoding', 
             fontsize=14, fontweight='bold', y=0.995)
plt.show()

## Summary

In [ ]:
print("="*70)
print("BERT DEEP ANALYSIS SUMMARY")
print("="*70)

print(f"\n1. CLASSIFICATION PERFORMANCE:")
print(f"   Accuracy: {accuracy:.2%}")
print(f"   True Positives (Real): {cm[0,0]} | False Positives: {cm[0,1]}")
print(f"   False Negatives (Fake): {cm[1,0]} | True Positives (Fake): {cm[1,1]}")

print(f"\n2. MODEL ARCHITECTURE:")
print(f"   Layers: {num_layers}")
print(f"   Attention Heads: {num_heads} per layer")
print(f"   Hidden Dimension: {token_embedding_dim}")
print(f"   Total Parameters: ~110M (BERT-base)")

print(f"\n3. ATTENTION MECHANISMS:")
print(f"   ✓ Multi-head self-attention in all {num_layers} layers")
print(f"   ✓ Scaled dot-product attention (scale: 1/sqrt({token_embedding_dim//num_heads}))")
print(f"   ✓ Learned query, key, value projections")
print(f"   ✓ Softmax normalization")

print(f"\n4. ATTENTION MASKING:")
print(f"   ✓ Padding tokens are masked (attention weight ≈ 0)")
print(f"   ✓ Valid tokens: 0-{int(val_attention_mask[0].sum().item())-1}")
print(f"   ✓ Padding tokens: {int(val_attention_mask[0].sum().item())}-{seq_len-1}")
print(f"   ✓ Prevents model from attending to irrelevant padding")

print(f"\n5. POSITIONAL ENCODING:")
print(f"   ✓ Sinusoidal positional encoding")
print(f"   ✓ Dimension: {token_embedding_dim}")
print(f"   ✓ Max sequence length: {max_pos}")
print(f"   ✓ No learned parameters (fixed encodings)")
print(f"   ✓ Added to token embeddings at input layer")

print(f"\n6. INFORMATION FLOW:")
print(f"   Input tokens → Token embedding + Positional encoding")
print(f"   → Layer 1: Multi-head attention (with masking) + Feed-forward")
print(f"   → Layers 2-12: Same structure, refining representations")
print(f"   → [CLS] token at output → Classification head")
print(f"   → Prediction (Real/Fake)")

print(f"\n" + "="*70)